# Module 6: Recommendation Impact on Basket Behavior

This notebook quantifies how the recommender changes real purchase patterns under the Module 4 temporal split.

Covered outputs:
- Category Expansion Rate (new categories purchased that appear in Top-5 recommendations)
- Margin Shift Index (train vs test Normalized_Margin)
- Basket Diversity Uplift (distinct commodities per basket)
- Hit-rate vs Discovery trade-off

Figures saved to reports/figures:
- basket_diversity_comparison.html
- margin_shift.html
- tradeoff_scatter.html
Scope: compare behavior in test vs train periods.
Use results to justify basket impact beyond offline metrics.

## Setup and Paths
This cell loads core libraries, resolves project paths, and defines small helper utilities used across the notebook.
It also ensures downstream cells share the same root paths.
If paths look wrong, update ROOT before running.

In [1]:
from pathlib import Path
import importlib.util
import sys

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

DATA_RAW = ROOT / 'data' / '01_raw'
DATA_PROCESSED = ROOT / 'data' / '02_processed'
REPORTS = ROOT / 'reports'
FIGURES = REPORTS / 'figures'
FIGURES.mkdir(parents=True, exist_ok=True)

def read_csv_or_none(path: Path):
    if not path.exists():
        return None
    frame = pd.read_csv(path)
    if list(frame.columns) == ['version https://git-lfs.github.com/spec/v1']:
        return None
    return frame

def package_available(name: str) -> bool:
    return importlib.util.find_spec(name) is not None

print('Project root:', ROOT)
print('plotly available:', package_available('plotly'))

Project root: C:\Users\dell\Downloads\New folder (9)\Data_Driven_Marketing_complete-journey
plotly available: True


## Load Data and Artifacts
This cell loads master transactions and Module 3 artifacts, then prepares the normalized margin lookup used in later metrics.
It also checks required Module 3 outputs are available.
Missing files will stop the notebook early for clarity.

In [2]:
from src.data_loader import load_or_build_master_transactions
from src.module4_validation import (
    build_temporal_holdout,
    build_variant_topk,
    make_variant_weights,
    build_popularity_baseline_topk,
)
from src.utility_scorer import DEFAULT_UTILITY_WEIGHTS, build_commodity_margin_table, prepare_margin_lookup
from src.basket_impact import (
    compute_category_expansion_table,
    summarize_category_expansion,
    compute_margin_shift_table,
    compute_basket_diversity,
    compute_hit_rate_table,
    build_tradeoff_table,
)

product_catalog = pd.read_csv(DATA_RAW / 'product.csv', usecols=['PRODUCT_ID', 'COMMODITY_DESC', 'BRAND'])

# Avoid cached parquet schema issues in master transaction artifacts.
master_tx_dir = DATA_PROCESSED
all_path = master_tx_dir / 'master_transactions_all.parquet'
organic_path = master_tx_dir / 'master_transactions_organic_only.parquet'
if all_path.exists() or organic_path.exists():
    master_tx_dir = master_tx_dir / 'notebook_tmp'
    master_tx_dir.mkdir(parents=True, exist_ok=True)

tx_all, tx_organic = load_or_build_master_transactions(DATA_RAW, master_tx_dir)
scored_candidates = read_csv_or_none(DATA_PROCESSED / 'candidate_set_module3_scored.csv')
commodity_margin = read_csv_or_none(DATA_PROCESSED / 'commodity_margin.csv')

if scored_candidates is None:
    raise FileNotFoundError('Module 3 candidate_set_module3_scored.csv is required before running Module 6.')
if commodity_margin is None:
    commodity_margin = build_commodity_margin_table(product_catalog)
    commodity_margin.to_csv(DATA_PROCESSED / 'commodity_margin.csv', index=False)

margin_lookup = prepare_margin_lookup(commodity_margin)

print('tx_all rows:', len(tx_all))
print('scored_candidates rows:', len(scored_candidates))
print('margin_lookup rows:', len(margin_lookup))
display(scored_candidates.head(5))

tx_all rows: 2595732
scored_candidates rows: 23544
margin_lookup rows: 308


,household_key,COMMODITY_DESC,seed_items,relevance_als,relevance_mba,Relevance,source,source_detail,snapshot_week,was_recently_purchased,...,Normalized_Margin,deal_sensitivity,is_active_campaign_period,item_is_promoted,Context,Utility,Relevance_Contribution,Uplift_Contribution,Margin_Contribution,Context_Contribution
0,1,POPCORN,BAKED BREAD/BUNS/ROLLS | CANDY - PACKAGED | CA...,0.947021,0.000000,0.947021,als,ALS,102,False,...,1.0,0.813953,False,True,0.5,0.900901,0.378808,0.247093,0.2,0.075
1,1,VEGETABLES - ALL OTHERS,BAKED BREAD/BUNS/ROLLS | CANDY - PACKAGED | CA...,0.000000,1.000000,1.000000,mba,MBA,102,False,...,1.0,0.813953,False,True,0.5,0.898837,0.400000,0.223837,0.2,0.075
2,1,BEEF,BAKED BREAD/BUNS/ROLLS | CANDY - PACKAGED | CA...,0.000000,0.796527,0.796527,mba,MBA,102,False,...,1.0,0.813953,False,True,0.5,0.837797,0.318611,0.244186,0.2,0.075
3,1,BLEACH,BAKED BREAD/BUNS/ROLLS | CANDY - PACKAGED | CA...,0.721154,0.000000,0.721154,als,ALS,102,False,...,1.0,0.813953,False,True,0.5,0.798927,0.288462,0.235465,0.2,0.075
4,1,CHRISTMAS SEASONAL,BAKED BREAD/BUNS/ROLLS | CANDY - PACKAGED | CA...,0.523417,0.000000,0.523417,als,ALS,102,False,...,1.0,0.813953,False,True,0.5,0.728553,0.209367,0.244186,0.2,0.075


## Build Holdout and Top-5 Lists
This cell creates the temporal split and builds the Chimera, relevance-only, and popularity top-5 recommendation sets.
Only eligible households with both train and test history are kept.
Top-5 lists are the basis for all impact metrics below.

In [3]:
split = build_temporal_holdout(tx_all)
eligible = split.eligible_households

chimera_top5 = build_variant_topk(scored_candidates, DEFAULT_UTILITY_WEIGHTS, top_k=5)
relevance_weights = make_variant_weights(relevance=1.0, uplift=0.0, margin=0.0, context=0.0)
relevance_top5 = build_variant_topk(scored_candidates, relevance_weights, top_k=5)
popularity_top5 = build_popularity_baseline_topk(
    train_history=split.train_history,
    eligible_households=eligible,
    train_items_by_user=split.train_items_by_user,
    top_k=5,
)

chimera_top5 = chimera_top5[chimera_top5['household_key'].isin(eligible)]
relevance_top5 = relevance_top5[relevance_top5['household_key'].isin(eligible)]
popularity_top5 = popularity_top5[popularity_top5['household_key'].isin(eligible)]

print('eligible households:', len(eligible))
print('chimera top5 rows:', len(chimera_top5))
print('relevance top5 rows:', len(relevance_top5))
print('popularity top5 rows:', len(popularity_top5))

eligible households: 2408
chimera top5 rows: 2395
relevance top5 rows: 2395
popularity top5 rows: 12040


## Category Expansion Rate
This cell compares how often each model recommends categories that appear as new purchases in the test period.
Higher expansion rate implies better discovery alignment.
We compare Chimera vs Popularity in the summary table.

In [4]:
chimera_expansion = compute_category_expansion_table(
    chimera_top5,
    split.train_items_by_user,
    split.test_items_by_user,
    top_k=5,
)
pop_expansion = compute_category_expansion_table(
    popularity_top5,
    split.train_items_by_user,
    split.test_items_by_user,
    top_k=5,
)

expansion_summary = pd.concat(
    [
        summarize_category_expansion(chimera_expansion, 'Chimera'),
        summarize_category_expansion(pop_expansion, 'Popularity'),
    ],
    ignore_index=True,
)

display(expansion_summary)

,Model,Households,Category_Expansion_Rate,Avg_New_Categories,Avg_New_Categories_Hit
0,Chimera,2408,0.053571,12.012043,0.068937
1,Popularity,2408,0.674003,12.012043,1.259551


## Margin Shift Index
This cell summarizes average margin in train vs test periods and visualizes the shift by treatment group.
Positive shifts indicate movement toward higher-margin mix.
The bar chart shows average margin per period.

In [5]:
margin_shift = compute_margin_shift_table(
    split.train_history,
    split.test_history,
    margin_lookup,
    eligible_households=eligible,
)

chimera_hits = compute_hit_rate_table(chimera_top5, split.test_items_by_user, top_k=5)
pop_hits = compute_hit_rate_table(popularity_top5, split.test_items_by_user, top_k=5)

def summarize_margin_shift(shift_table: pd.DataFrame, hit_table: pd.DataFrame, label: str) -> pd.DataFrame:
    merged = shift_table.merge(hit_table[['household_key', 'has_hit']], on='household_key', how='left')
    merged = merged[merged['has_hit'].fillna(False)]
    return pd.DataFrame(
        [
            {
                'treatment': label,
                'train_margin': merged['train_margin'].mean(),
                'test_margin': merged['test_margin'].mean(),
                'margin_shift': merged['margin_shift'].mean(),
                'households': len(merged),
            }
        ]
    )

margin_summary = pd.concat(
    [
        summarize_margin_shift(margin_shift, chimera_hits, 'Chimera'),
        summarize_margin_shift(margin_shift, pop_hits, 'Popularity'),
    ],
    ignore_index=True,
)

display(margin_summary)

margin_plot = margin_summary.melt(
    id_vars=['treatment', 'households', 'margin_shift'],
    value_vars=['train_margin', 'test_margin'],
    var_name='period',
    value_name='avg_margin',
)
margin_plot['period'] = margin_plot['period'].str.replace('_margin', '', regex=False)

fig_margin = px.bar(
    margin_plot,
    x='treatment',
    y='avg_margin',
    color='period',
    barmode='group',
    title='Average Normalized Margin: Train vs Test',
)
fig_margin.write_html(FIGURES / 'margin_shift.html')
fig_margin.show()

C:\Users\dell\AppData\Local\Temp\ipykernel_54300\2155503949.py:13: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  merged = merged[merged['has_hit'].fillna(False)]


,treatment,train_margin,test_margin,margin_shift,households
0,Chimera,0.901613,0.912415,0.010802,363
1,Popularity,0.900161,0.914841,0.014679,1623


## Basket Diversity Uplift
This cell measures distinct commodities per basket in the test period and plots the distribution by treatment.
Distribution view highlights whether baskets become more diverse.
We filter to households with at least one hit.

In [6]:
basket_diversity, household_diversity = compute_basket_diversity(split.test_history)

chimera_group = household_diversity.merge(
    chimera_hits[['household_key', 'has_hit']], on='household_key', how='left'
)
chimera_group = chimera_group[chimera_group['has_hit'].fillna(False)].assign(treatment='Chimera')

pop_group = household_diversity.merge(
    pop_hits[['household_key', 'has_hit']], on='household_key', how='left'
)
pop_group = pop_group[pop_group['has_hit'].fillna(False)].assign(treatment='Popularity')

diversity_frame = pd.concat([chimera_group, pop_group], ignore_index=True)

fig_div = px.histogram(
    diversity_frame,
    x='avg_distinct_commodities',
    color='treatment',
    barmode='overlay',
    nbins=25,
    title='Basket Diversity in Test Period',
)
fig_div.update_layout(xaxis_title='Avg distinct commodities per basket')
fig_div.write_html(FIGURES / 'basket_diversity_comparison.html')
fig_div.show()

print('chimera households with hits:', chimera_group['household_key'].nunique())
print('popularity households with hits:', pop_group['household_key'].nunique())

C:\Users\dell\AppData\Local\Temp\ipykernel_54300\4050674598.py:6: FutureWarning:

Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`

C:\Users\dell\AppData\Local\Temp\ipykernel_54300\4050674598.py:11: FutureWarning:

Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`



chimera households with hits: 363
popularity households with hits: 1623


## Hit-Rate vs Discovery Summary
This cell produces a simple table comparing hit-rate and discovery between Chimera and relevance-only models.
Avg hit-rate captures relevance while new categories capture discovery.
Use both to judge trade-offs across models.

In [10]:
chimera_tradeoff = build_tradeoff_table(
    chimera_top5,
    split.train_items_by_user,
    split.test_items_by_user,
    model_label='Chimera',
    top_k=5,
 )
relevance_tradeoff = build_tradeoff_table(
    relevance_top5,
    split.train_items_by_user,
    split.test_items_by_user,
    model_label='Relevance Only',
    top_k=5,
 )

tradeoff = pd.concat([chimera_tradeoff, relevance_tradeoff], ignore_index=True)

summary = (
    tradeoff.groupby('model', as_index=False)
    .agg(
        households=('household_key', 'nunique'),
        avg_hit_rate=('hit_rate', 'mean'),
        avg_hits=('hits', 'mean'),
        avg_new_categories=('new_categories_purchased', 'mean'),
        median_new_categories=('new_categories_purchased', 'median'),
    )
    .sort_values('avg_hit_rate', ascending=False)
 )

display(summary)

,model,households,avg_hit_rate,avg_hits,avg_new_categories,median_new_categories
1,Relevance Only,479,0.372860,1.864301,11.609603,10.0
0,Chimera,479,0.348643,1.743215,11.609603,10.0


## Interpretation Notes

- Category expansion captures whether recommendations align with new categories purchased in the test period.
- Margin shift compares normalized margin in the test period vs training for the same households.
- Basket diversity uses distinct commodities per basket as a proxy for variety.
- Trade-off scatter contrasts hit-rate against new categories purchased to show discovery impact.
Consider sample size and eligibility when comparing models.
Use these signals alongside Module 4 precision metrics.